# Car Market Analysis - Car Dekho Dataset

**Technologies:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** Car Dekho used-car listings (301 records, 2003-2018)

## 0. Imports & Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

COLORS = {
    'primary': '#1565C0', 'accent1': '#00ACC1',
    'accent2': '#43A047', 'accent3': '#FB8C00', 'accent4': '#E53935',
}
PALETTE = [COLORS['primary'], COLORS['accent1'], COLORS['accent2'],
           COLORS['accent3'], COLORS['accent4']]

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.color': '#BDBDBD',
    'figure.dpi': 120,
})
print('Setup complete')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('data/car_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
df.describe()

## 2. Data Cleaning & Feature Engineering

In [ ]:
df['Car_Age']          = 2024 - df['Year']
df['Price_Drop']       = df['Present_Price'] - df['Selling_Price']
df['Depreciation_Pct'] = (df['Price_Drop'] / df['Present_Price']) * 100

print('Car_Age range    :', df['Car_Age'].min(), '-', df['Car_Age'].max(), 'years')
print('Avg depreciation :', round(df['Depreciation_Pct'].mean(), 1), '%')
print('Avg selling price: Rs', round(df['Selling_Price'].mean(), 2), 'Lakhs')
df.head()

## 3. Exploratory Data Analysis

### 3.1 Fuel Type Distribution

In [ ]:
fuel_counts = df['Fuel_Type'].value_counts()
print(fuel_counts)

wedge_colors = [COLORS['primary'], COLORS['accent1'], COLORS['accent2']]
fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
wedges, texts, autotexts = ax.pie(
    fuel_counts.values, labels=fuel_counts.index,
    autopct='%1.1f%%', colors=wedge_colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for t in texts: t.set_fontsize(12)
for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold'); t.set_color('white')
ax.set_title('Fuel Type Distribution', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot1_fuel_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 3.2 Avg Selling Price by Fuel Type

In [ ]:
avg_price = df.groupby('Fuel_Type')['Selling_Price'].mean().sort_values(ascending=False)
print(avg_price.round(2))

fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
bars = ax.bar(avg_price.index, avg_price.values,
              color=wedge_colors[:len(avg_price)], edgecolor='white', linewidth=1.5, width=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'Rs {bar.get_height():.1f}L', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Avg Selling Price by Fuel Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Selling Price (Lakhs)', fontsize=11)
ax.set_xlabel('Fuel Type', fontsize=11)
ax.set_ylim(0, avg_price.max() * 1.2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot2_avg_price_fuel.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 3.3 Selling Price by Transmission Type

In [ ]:
manual_avg = df[df['Transmission'] == 'Manual']['Selling_Price'].mean()
auto_avg   = df[df['Transmission'] == 'Automatic']['Selling_Price'].mean()
print(f'Manual avg    : Rs {manual_avg:.2f}L')
print(f'Automatic avg : Rs {auto_avg:.2f}L  (Automatic is {auto_avg/manual_avg:.1f}x more expensive)')

trans_groups = [df[df['Transmission'] == t]['Selling_Price'].values for t in ['Manual', 'Automatic']]
fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
bp = ax.boxplot(trans_groups, labels=['Manual', 'Automatic'], patch_artist=True,
    medianprops=dict(color=COLORS['primary'], linewidth=2.5),
    whiskerprops=dict(color='#666'), capprops=dict(color='#666'))
bp['boxes'][0].set_facecolor('#E3F2FD')
bp['boxes'][1].set_facecolor('#E0F7FA')
ax.set_title('Selling Price by Transmission Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Selling Price (Lakhs)', fontsize=11)
ax.set_xlabel('Transmission Type', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot3_price_transmission.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 3.4 Selling Price vs Kms Driven

In [ ]:
fuel_palette = {'Petrol': COLORS['primary'], 'Diesel': COLORS['accent1'], 'CNG': COLORS['accent2']}
fig, ax = plt.subplots(figsize=(8, 5), facecolor='white')
for fuel, grp in df.groupby('Fuel_Type'):
    ax.scatter(grp['Kms_Driven']/1000, grp['Selling_Price'],
               label=fuel, color=fuel_palette.get(fuel, 'gray'),
               alpha=0.65, s=60, edgecolors='white', linewidth=0.5)
ax.set_title('Selling Price vs Kms Driven', fontsize=14, fontweight='bold')
ax.set_xlabel('Kms Driven (Thousands)', fontsize=11)
ax.set_ylabel('Selling Price (Lakhs)', fontsize=11)
ax.legend(fontsize=10, framealpha=0.8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot4_price_vs_kms.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 3.5 Year-wise Avg Selling Price Trend

In [ ]:
year_price = df.groupby('Year')['Selling_Price'].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 5), facecolor='white')
ax.plot(year_price['Year'], year_price['Selling_Price'],
        color=COLORS['primary'], marker='o', linewidth=2.5, markersize=8,
        markerfacecolor=COLORS['accent1'], markeredgecolor='white', markeredgewidth=1.5)
ax.fill_between(year_price['Year'], year_price['Selling_Price'], alpha=0.15, color=COLORS['primary'])
for _, row in year_price.iterrows():
    ax.text(row['Year'], row['Selling_Price'] + 0.3,
            f"Rs {row['Selling_Price']:.1f}L", ha='center', fontsize=8, color='#444')
ax.set_title('Year-wise Avg Selling Price Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Manufacturing Year', fontsize=11)
ax.set_ylabel('Avg Selling Price (Lakhs)', fontsize=11)
ax.set_xticks(year_price['Year'])
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot5_year_trend.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 3.6 Feature Correlation Heatmap

In [ ]:
corr_cols = ['Selling_Price', 'Present_Price', 'Kms_Driven', 'Car_Age', 'Owner']
corr      = df[corr_cols].corr()
labels    = ['Selling\nPrice', 'Present\nPrice', 'Kms\nDriven', 'Car\nAge', 'Owner']

fig, ax = plt.subplots(figsize=(7, 5.5), facecolor='white')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.8},
            xticklabels=labels, yticklabels=labels, annot_kws={'size': 11})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot6_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Key finding: Selling_Price and Present_Price are highly correlated (0.88)')

### 3.7 Avg Selling Price by Seller Type

In [ ]:
seller_avg = df.groupby('Seller_Type')['Selling_Price'].mean()
seller_cnt = df.groupby('Seller_Type')['Selling_Price'].count()

fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
x    = np.arange(len(seller_avg))
bars = ax.bar(x, seller_avg.values, color=[COLORS['primary'], COLORS['accent1']],
              edgecolor='white', linewidth=1.5, width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(seller_avg.index, fontsize=12)
ax.set_ylabel('Avg Selling Price (Lakhs)', fontsize=11)
ax.set_title('Avg Selling Price by Seller Type', fontsize=14, fontweight='bold')
for bar, cnt in zip(bars, seller_cnt.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'Rs {bar.get_height():.1f}L\n(n={cnt})', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, seller_avg.max() * 1.35)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot7_seller_type.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 3.8 Car Age vs Selling Price

In [ ]:
corr_age = df['Car_Age'].corr(df['Selling_Price'])
print(f'Pearson correlation (Car Age vs Selling Price): {corr_age:.3f}')

z      = np.polyfit(df['Car_Age'], df['Selling_Price'], 1)
p      = np.poly1d(z)
x_line = np.linspace(df['Car_Age'].min(), df['Car_Age'].max(), 100)

fig, ax = plt.subplots(figsize=(8, 5), facecolor='white')
ax.scatter(df['Car_Age'], df['Selling_Price'],
           color=COLORS['primary'], alpha=0.55, s=55, edgecolors='white', linewidth=0.5)
ax.plot(x_line, p(x_line), color=COLORS['accent3'], linewidth=2.5, linestyle='--', label='Trend Line')
ax.set_title('Car Age vs Selling Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Car Age (Years)', fontsize=11)
ax.set_ylabel('Selling Price (Lakhs)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot8_age_vs_price.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### 3.9 Top 10 Most Listed Car Models

In [ ]:
top_cars = df['Car_Name'].value_counts().head(10)
print(top_cars)

fig, ax = plt.subplots(figsize=(8, 5), facecolor='white')
bars = ax.barh(top_cars.index[::-1], top_cars.values[::-1],
               color=[PALETTE[i % len(PALETTE)] for i in range(len(top_cars))],
               edgecolor='white', linewidth=1)
for bar in bars:
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontsize=10, fontweight='bold')
ax.set_title('Top 10 Most Listed Car Models', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Listings', fontsize=11)
ax.set_xlim(0, top_cars.max() * 1.18)
ax.set_yticklabels([n.upper() for n in top_cars.index[::-1]], fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot9_top_10_cars.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Key Insights Summary

In [ ]:
print('=' * 60)
print('  KEY INSIGHTS - Car Market Analysis')
print('=' * 60)
print(f'  Total records    : {len(df)}')
print(f'  Year range       : {df["Year"].min()} - {df["Year"].max()}')
print(f'  Avg sell price   : Rs {df["Selling_Price"].mean():.2f} Lakhs')
print(f'  Price range      : Rs {df["Selling_Price"].min():.2f}L to Rs {df["Selling_Price"].max():.2f}L')
print(f'  Avg kms driven   : {df["Kms_Driven"].mean():.0f} km')
print(f'  Avg depreciation : {df["Depreciation_Pct"].mean():.1f}%')
print()
print(f'  Petrol share     : {fuel_counts.get("Petrol",0)/len(df)*100:.1f}%')
print(f'  Diesel share     : {fuel_counts.get("Diesel",0)/len(df)*100:.1f}%')
print(f'  CNG share        : {fuel_counts.get("CNG",0)/len(df)*100:.1f}%')
print()
print(f'  Manual avg price : Rs {manual_avg:.2f}L')
print(f'  Automatic avg    : Rs {auto_avg:.2f}L')
print()
print(f'  Dealer avg price : Rs {seller_avg.get("Dealer",0):.2f}L')
print(f'  Individual avg   : Rs {seller_avg.get("Individual",0):.2f}L')
print()
print(f'  Top car model    : {top_cars.index[0].upper()} with {top_cars.iloc[0]} listings')
print(f'  Age-Price corr   : {corr_age:.3f} (older => cheaper)')
print('=' * 60)